# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [10]:
%reload_ext dotenv
%dotenv 

In [11]:
import dask.dataframe as dd

In [33]:
import os
os.chdir('/Users/zoeackah/production/05_src')  # Replace with the correct path
print("Current Working Directory:", os.getcwd())

Current Working Directory: /Users/zoeackah/production/05_src


In [34]:
print(os.path.join(os.getenv('PRICE_DATA')))

../05_src/data/prices/


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [36]:
import os
print("PRICE_DATA:", os.getenv('PRICE_DATA'))
from glob import glob

parquet_files = glob(os.path.join(os.getenv('PRICE_DATA'), "**/*.parquet"), recursive= True)

print(f"Found {len(parquet_files)} parquet files")
print("First few files:")
for f in parquet_files[:5]:
    print(f)


PRICE_DATA: ../05_src/data/prices/
Found 2918 parquet files
First few files:
../05_src/data/prices/BKTI/BKTI_2012/part.0.parquet
../05_src/data/prices/BKTI/BKTI_2012/part.1.parquet
../05_src/data/prices/BKTI/BKTI_2015/part.0.parquet
../05_src/data/prices/BKTI/BKTI_2015/part.1.parquet
../05_src/data/prices/BKTI/BKTI_2014/part.0.parquet


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [54]:
# Thank you Dmytro B. for explaining this so carefully
ddf = dd.read_parquet(parquet_files, engine='pyarrow')

ddf_result = ddf.groupby('ticker').apply(lambda x: x.assign(Close_lag_1 = x['Close'].shift(1),
                            returns = x['Close']/x['Close'].shift(1) - 1)).compute()
print(ddf_result.head())
                            
                      

/var/folders/0c/3b2dkdy1209_gm_z1sjrc81c0000gn/T/ipykernel_96855/1484695778.py:4: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf_result = ddf.groupby('ticker').apply(lambda x: x.assign(Close_lag_1 = x['Close'].shift(1),


                 Date       Open       High        Low      Close  Adj Close  \
ticker                                                                         
JJM    760 2011-01-03  48.139999  48.139999  46.500000  46.970001  46.970001   
       761 2011-01-04  47.040001  47.040001  46.160000  46.830002  46.830002   
       762 2011-01-05  47.689999  47.689999  45.830002  46.660000  46.660000   
       763 2011-01-06  46.480000  46.880001  46.160000  46.330002  46.330002   
       764 2011-01-07  46.180000  46.330002  45.799999  45.799999  45.799999   

             Volume   source ticker  Year  Close_lag_1   returns  
ticker                                                            
JJM    760  26300.0  JJM.csv    JJM  2011          NaN       NaN  
       761  16400.0  JJM.csv    JJM  2011    46.970001 -0.002981  
       762  98700.0  JJM.csv    JJM  2011    46.830002 -0.003630  
       763  17000.0  JJM.csv    JJM  2011    46.660000 -0.007072  
       764   2800.0  JJM.csv    JJM  

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [56]:
# It is already a pandas DataFrame as seen above. 

ddf_result

Date       Open       High        Low      Close  Adj Close  \
ticker                                                                          
JJM    760  2011-01-03  48.139999  48.139999  46.500000  46.970001  46.970001   
       761  2011-01-04  47.040001  47.040001  46.160000  46.830002  46.830002   
       762  2011-01-05  47.689999  47.689999  45.830002  46.660000  46.660000   
       763  2011-01-06  46.480000  46.880001  46.160000  46.330002  46.330002   
       764  2011-01-07  46.180000  46.330002  45.799999  45.799999  45.799999   
...                ...        ...        ...        ...        ...        ...   
CASY   4715 2002-06-26  11.700000  11.980000  11.650000  11.800000   9.898915   
       4716 2002-06-27  11.800000  11.870000  11.660000  11.810000   9.907303   
       4717 2002-06-28  11.860000  12.220000  11.840000  12.040000  10.100246   
       4718 2002-07-01  11.990000  12.120000  11.850000  12.120000  10.167357   
       4719 2002-07-02  11.900000  11.980000  11.450000  11.500000   9.647246   

               Volume    source ticker  Year  Close_lag_1   returns  
ticker                                                               
JJM    760    26300.0   JJM.csv    JJM  2011          NaN       NaN  
       761    16400.0   JJM.csv    JJM  2011    46.970001 -0.002981  
       762    98700.0   JJM.csv    JJM  2011    46.830002 -0.003630  
       763    17000.0   JJM.csv    JJM  2011    46.660000 -0.007072  
       764     2800.0   JJM.csv    JJM  2011    46.330002 -0.011440  
...               ...       ...    ...   ...          ...       ...  
CASY   4715  217400.0  CASY.csv   CASY  2002    11.990000 -0.015847  
       4716  396200.0  CASY.csv   CASY  2002    11.800000  0.000847  
       4717  323300.0  CASY.csv   CASY  2002    11.810000  0.019475  
       4718  259800.0  CASY.csv   CASY  2002    12.040000  0.006645  
       4719   85200.0  CASY.csv   CASY  2002    12.120000 -0.051155  

[334064 rows x 12 columns]

In [57]:
ddf = dd.read_parquet(parquet_files, engine='pyarrow')

ddf_result = ddf.groupby('ticker').apply(lambda x: x.assign(Close_lag_1=x['Close'].shift(1),
                                                        returns=x['Close'] / x['Close'].shift(1) - 1)).compute()

ddf_result = ddf_result.assign(moving_average=ddf_result['returns'].rolling(window=10).mean())

/var/folders/0c/3b2dkdy1209_gm_z1sjrc81c0000gn/T/ipykernel_96855/2536807826.py:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  ddf_result = ddf.groupby('ticker').apply(lambda x: x.assign(Close_lag_1=x['Close'].shift(1),


In [58]:
ddf_result

Date       Open       High        Low      Close  Adj Close  \
ticker                                                                          
JJM    1137 2012-07-02  33.169998  33.169998  32.919998  33.119999  33.119999   
       1138 2012-07-03  34.009998  34.200001  33.950001  34.040001  34.040001   
       1139 2012-07-05  33.200001  33.549999  33.200001  33.400002  33.400002   
       1140 2012-07-06  32.869999  32.910000  32.459999  32.459999  32.459999   
       1141 2012-07-09  32.709999  32.880001  32.689999  32.799999  32.799999   
...                ...        ...        ...        ...        ...        ...   
CASY   2700 1994-06-27   5.562500   5.562500   5.312500   5.437500   4.334043   
       2701 1994-06-28   5.437500   5.562500   5.312500   5.375000   4.284226   
       2702 1994-06-29   5.562500   5.750000   5.437500   5.625000   4.483496   
       2703 1994-06-30   5.687500   5.687500   5.562500   5.687500   4.533312   
       2704 1994-07-01   5.687500   5.750000   5.625000   5.750000   4.583125   

               Volume    source ticker  Year  Close_lag_1   returns  \
ticker                                                                
JJM    1137   16900.0   JJM.csv    JJM  2012          NaN       NaN   
       1138    6200.0   JJM.csv    JJM  2012    33.119999  0.027778   
       1139    3600.0   JJM.csv    JJM  2012    34.040001 -0.018801   
       1140    2600.0   JJM.csv    JJM  2012    33.400002 -0.028144   
       1141   72100.0   JJM.csv    JJM  2012    32.459999  0.010474   
...               ...       ...    ...   ...          ...       ...   
CASY   2700  102600.0  CASY.csv   CASY  1994     5.562500 -0.022472   
       2701  132800.0  CASY.csv   CASY  1994     5.437500 -0.011494   
       2702   85400.0  CASY.csv   CASY  1994     5.375000  0.046512   
       2703   43000.0  CASY.csv   CASY  1994     5.625000  0.011111   
       2704   55000.0  CASY.csv   CASY  1994     5.687500  0.010989   

             moving_average  
ticker                       
JJM    1137             NaN  
       1138             NaN  
       1139             NaN  
       1140             NaN  
       1141             NaN  
...                     ...  
CASY   2700       -0.009574  
       2701       -0.012807  
       2702       -0.011217  
       2703       -0.009116  
       2704       -0.006517  

[334064 rows x 13 columns]

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return? 
Because you can't change anything in a parquet file. You have to convert it in order to edit - you need to add the close_lag_1, returns and moving_average columns to save your results. Parquet files are for easy and efficient storage - you read them, you write to them but you have to 'do stuff' in parquet files. 
+ Would it have been better to do it in Dask? Why?
Wow what I nightmare. I tried to run it in Dask but when I did the output didn't match when compared to my pandas frame, in fact it was totally wrong. I think Dask partitions data to process in in parallel and so it messes up things that exist in time like yesterdays closing price, it doesn't maintain and order that indicated it happened BEFORE today's closing price. I think it partitions BEFORE if groups by ticker so that means the output is essentially trash. Yesterday's price is in another partition so it can't find it to compare to today's price, and it doesn't maintain temporal things well. 

(1 pt)

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.